# Ablation Study 1 — Message Passing Depth

**Question:** How does the number of GNN layers affect performance?  
**Hypothesis:** Too few layers → under-reaches the full molecule. Too many → over-smoothing collapses node embeddings.  
**Models:** GCN, GAT, GATv2  
**Variable:** `num_layers` ∈ {2, 6, 12}  
**Fixed:** All other hyperparameters held constant across models and depths.

**Output directory:** `MyDrive/Ablation/Study1_LayerDepth/`

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, subprocess, sys
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '')  # ← auto-detect
print(f'Using torch={torch_version}, cuda_tag={cuda_tag}')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch-scatter', 'torch-sparse',
    '-f', f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)
print('PyG installed.')

In [ ]:
# ── Cell 3: Clone repo & set paths ────────────────────────────────────────────
import os, sys

REPO_URL  = 'https://github.com/amanikonda123/DL-Final-Project.git'  # ← update this
REPO_DIR  = '/content/gnn_project'
DRIVE_OUT = '/content/drive/MyDrive/Ablation/Study1_LayerDepth'
DATA_ROOT = '/content/qm9_data'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Output subdirectories
for sub in ['checkpoints', 'logs', 'results', 'plots']:
    os.makedirs(f'{DRIVE_OUT}/{sub}', exist_ok=True)

print(f'Output root: {DRIVE_OUT}')

In [ ]:
# ── Cell 4: USER CONFIG ───────────────────────────────────────────────────────
# Edit these values directly. No YAML reading.

# ---------- Training ----------
EPOCHS      = 300
PATIENCE    = 30
BATCH_SIZE  = 128
LR          = 5e-4

# ---------- Model (fixed across all depth runs) ----------
HIDDEN_DIM  = 256
DROPOUT     = 0.0
HEADS       = 8          # GAT / GATv2 only
FEATURE_MODE = 'topology' # 'topology' or 'full'

# ---------- Ablation variable ----------
LAYER_DEPTHS = [2, 6, 12]   # The sweep
MODELS       = ['gcn', 'gat', 'gatv2']

# ---------- Dataset ----------
TARGET_IDX  = 7     # U0 — internal energy at 0K
SEED        = 42
SPLIT       = [0.8, 0.1, 0.1]

print('Config:')
print(f'  epochs={EPOCHS}, patience={PATIENCE}, batch={BATCH_SIZE}, lr={LR}')
print(f'  hidden={HIDDEN_DIM}, dropout={DROPOUT}, heads={HEADS}')
print(f'  feature_mode={FEATURE_MODE}')
print(f'  layer depths: {LAYER_DEPTHS}')
print(f'  models: {MODELS}')

In [ ]:
# ── Cell 5: Load data once ────────────────────────────────────────────────────
from data.loader import get_dataloaders

BASE_CONFIG = {
    'dataset':  {'target': TARGET_IDX, 'split': SPLIT, 'seed': SEED, 'feature_mode': FEATURE_MODE},
    'training': {'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'lr': LR},
}

train_loader, val_loader, test_loader, normalizer = get_dataloaders(BASE_CONFIG, root=DATA_ROOT)
print('Data loaded.')

In [ ]:
# ── Cell 6: Training loop ─────────────────────────────────────────────────────
import copy, csv, time
import torch
import torch.nn.functional as F
from tqdm import tqdm
from data.features import select_features, get_feature_dims
from models import build_model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def mae(pred, target):
    return (pred - target).abs().mean().item()


def run_epoch(model, loader, optimizer, device, normalizer, feature_mode, model_name, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = select_features(batch, mode=feature_mode)
            batch = batch.to(device)
            if model_name == 'gatv2':
                pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            else:
                pred = model(batch.x, batch.edge_index, batch.batch)
            target = batch.y.view(-1)
            loss = F.mse_loss(pred, target)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            all_preds.append(normalizer.denormalize(pred.detach().cpu()))
            all_targets.append(normalizer.denormalize(target.detach().cpu()))
    n = sum(t.size(0) for t in all_targets)
    return total_loss / n, mae(torch.cat(all_preds), torch.cat(all_targets))


def train_run(model_name, num_layers, feature_mode, run_id):
    """
    Train one model/depth combination.
    Returns dict with best_val_mae, history, checkpoint_path.
    """
    feature_dims = get_feature_dims(feature_mode)
    cfg = {
        'dataset':  {'target': TARGET_IDX, 'split': SPLIT, 'seed': SEED, 'feature_mode': feature_mode},
        'training': {'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'lr': LR},
        'model': {
            'hidden_dim': HIDDEN_DIM,
            'num_layers': num_layers,
            'dropout':    DROPOUT,
            'heads':      HEADS,
        }
    }

    model = build_model(model_name, cfg, feature_dims=feature_dims).to(DEVICE)
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [{run_id}] {model_name} layers={num_layers} | params: {param_count:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    ckpt_path = f'{DRIVE_OUT}/checkpoints/{run_id}_best.pt'
    log_path  = f'{DRIVE_OUT}/logs/{run_id}_history.csv'

    best_val_mae, best_state, patience_ctr = float('inf'), None, 0
    history = []

    with open(log_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writeheader()

    pbar = tqdm(range(1, EPOCHS + 1), desc=f'{run_id}', leave=True)
    for epoch in pbar:
        tr_loss, _ = run_epoch(model, train_loader, optimizer, DEVICE, normalizer, feature_mode, model_name, train=True)
        vl_loss, vl_mae = run_epoch(model, val_loader, None, DEVICE, normalizer, feature_mode, model_name, train=False)
        lr_now = optimizer.param_groups[0]['lr']

        row = {'epoch': epoch, 'train_loss': f'{tr_loss:.6f}', 'val_loss': f'{vl_loss:.6f}',
               'val_mae': f'{vl_mae:.6f}', 'lr': f'{lr_now:.2e}'}
        history.append(row)
        with open(log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writerow(row)

        pbar.set_postfix(val_mae=f'{vl_mae:.4f}')

        if vl_mae < best_val_mae:
            best_val_mae = vl_mae
            best_state   = copy.deepcopy(model.state_dict())
            patience_ctr = 0
            torch.save(best_state, ckpt_path)  # save immediately on improvement
        else:
            patience_ctr += 1

        if patience_ctr >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

    print(f'  [{run_id}] best val MAE = {best_val_mae:.4f} Ha')
    return {'run_id': run_id, 'model': model_name, 'num_layers': num_layers,
            'best_val_mae': best_val_mae, 'best_epoch': min(range(len(history)), key=lambda i: float(history[i]['val_mae'])) + 1,
            'params': param_count, 'checkpoint': ckpt_path}


print('Training functions defined.')

In [ ]:
# ── Cell 7: Run all ablation combinations ────────────────────────────────────
import pandas as pd

results = []
results_path = f'{DRIVE_OUT}/results/study1_results.csv'

total = len(MODELS) * len(LAYER_DEPTHS)
done  = 0

for model_name in MODELS:
    for num_layers in LAYER_DEPTHS:
        done += 1
        run_id = f'{model_name}_L{num_layers}'
        print(f'\n[{done}/{total}] Running {run_id} ...')
        result = train_run(model_name, num_layers, FEATURE_MODE, run_id)
        results.append(result)

        # Save incrementally — survives crash
        pd.DataFrame(results).to_csv(results_path, index=False)
        print(f'  Results saved → {results_path}')

df = pd.DataFrame(results)
print('\n── Study 1 Results ──')
print(df[['run_id', 'model', 'num_layers', 'best_val_mae', 'best_epoch', 'params']].to_string(index=False))

In [ ]:
# ── Cell 8: CVPR-style plot ───────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

# CVPR half-column: 3.5in wide, font matching LaTeX
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'font.size':         9,
    'axes.titlesize':    9,
    'axes.labelsize':    9,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'legend.fontsize':   8,
    'figure.dpi':        300,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

# IBM colorblind-safe palette
COLORS = {'gcn': '#648FFF', 'gat': '#FE6100', 'gatv2': '#DC267F'}
MARKERS = {'gcn': 'o', 'gat': 's', 'gatv2': '^'}
LABELS  = {'gcn': 'GCN', 'gat': 'GAT', 'gatv2': 'GATv2'}

df = pd.read_csv(f'{DRIVE_OUT}/results/study1_results.csv')

fig, ax = plt.subplots(figsize=(3.5, 2.6))

for model_name in MODELS:
    sub = df[df['model'] == model_name].sort_values('num_layers')
    ax.plot(
        sub['num_layers'], sub['best_val_mae'],
        color=COLORS[model_name], marker=MARKERS[model_name],
        linewidth=1.4, markersize=5, label=LABELS[model_name]
    )

ax.set_xlabel('Number of Layers')
ax.set_ylabel('Val MAE (Ha)')
ax.set_title('Effect of Message Passing Depth on U0 Prediction')
ax.set_xticks(LAYER_DEPTHS)
ax.legend(frameon=False)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
fig.tight_layout(pad=0.4)

plot_path = f'{DRIVE_OUT}/plots/study1_layer_depth.pdf'
fig.savefig(plot_path, format='pdf', bbox_inches='tight')
fig.savefig(plot_path.replace('.pdf', '.png'), format='png', bbox_inches='tight', dpi=300)
plt.show()
print(f'Plot saved → {plot_path}')

In [ ]:
# ── Cell 9: Learning curve plots (val MAE per epoch, per run) ─────────────────
import os, pandas as pd, matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(MODELS), figsize=(3.5 * len(MODELS), 2.6), sharey=True)

DEPTH_COLORS = {2: '#648FFF', 6: '#FE6100', 12: '#DC267F'}

for ax, model_name in zip(axes, MODELS):
    for num_layers in LAYER_DEPTHS:
        run_id   = f'{model_name}_L{num_layers}'
        log_path = f'{DRIVE_OUT}/logs/{run_id}_history.csv'
        if not os.path.exists(log_path):
            continue
        hist = pd.read_csv(log_path)
        ax.plot(hist['epoch'], hist['val_mae'].astype(float),
                color=DEPTH_COLORS[num_layers], linewidth=1.2, label=f'L={num_layers}')
    ax.set_title(LABELS[model_name])
    ax.set_xlabel('Epoch')
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel('Val MAE (Ha)')
fig.suptitle('Learning Curves by Depth', fontsize=9)
fig.tight_layout(pad=0.4)

curve_path = f'{DRIVE_OUT}/plots/study1_learning_curves.pdf'
fig.savefig(curve_path, format='pdf', bbox_inches='tight')
fig.savefig(curve_path.replace('.pdf', '.png'), format='png', bbox_inches='tight', dpi=300)
plt.show()
print(f'Learning curves saved → {curve_path}')